In [1]:
!pip install lime
!pip install kagglehub transformers torch -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=25117fc19fafaeebb42de98adfb615c24e4769104ed2460554645c753226a37c
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [2]:
import pandas as pd
import numpy as np
import pickle
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub


In [3]:
path = kagglehub.dataset_download("shivamb/real-or-fake-fake-jobposting-prediction")
df = pd.read_csv(f"{path}/fake_job_postings.csv")

print(f"✅ Dataset loaded: {df.shape}")
print(f"   Fraud rate: {df['fraudulent'].mean()*100:.2f}%")


100%|██████████| 16.1M/16.1M [00:00<00:00, 91.7MB/s]

Extracting files...


✅ Dataset loaded: (17880, 18)
   Fraud rate: 4.84%


In [6]:
# Text preprocessing
import nltk
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if pd.isna(text):
        return ""
    tokens = word_tokenize(str(text).lower())
    tokens = [token for token in tokens if token.isalpha() and token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(tokens)

# Combine text fields (same as your training)
def combine_text_fields(row):
    fields = ['title', 'location', 'company_profile', 'description',
              'requirements', 'benefits', 'required_experience',
              'required_education', 'industry', 'function']
    text_parts = []
    for field in fields:
        if pd.notna(row.get(field)):
            text_parts.append(str(row[field]))
    return ' '.join(text_parts) if text_parts else "unknown job"

print("Combining text fields...")
df['combined_text'] = df.apply(combine_text_fields, axis=1)

print("Preprocessing text...")
df['text_processed'] = df['combined_text'].apply(preprocess_text)

# Features
df['location_fraud_ratio'] = df.groupby('location')['fraudulent'].transform('mean').fillna(0.05)
df['character_count'] = df['combined_text'].str.len()

# Create feature set
X = df[['text_processed', 'telecommuting', 'has_company_logo',
         'has_questions', 'location_fraud_ratio', 'character_count']]
y = df['fraudulent'].values

# Train/test split (same random state as your training!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"✅ Data prepared")
print(f"   Train: {X_train.shape[0]} samples ({y_train.mean()*100:.2f}% fraud)")
print(f"   Test:  {X_test.shape[0]} samples ({y_test.mean()*100:.2f}% fraud)")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Combining text fields...
Preprocessing text...
✅ Data prepared
   Train: 12516 samples (4.84% fraud)
   Test:  5364 samples (4.85% fraud)


In [13]:
# Remove existing directory if it exists
import shutil
import os

if os.path.exists('job-postings-fraud'):
    shutil.rmtree('job-postings-fraud')
    print("🗑️ Removed existing directory")

!git clone https://github.com/tommygarner/job-postings-fraud.git
models_path = "job-postings-fraud/models/"
print(f"Cloned repo, models at: {models_path}")

🗑️ Removed existing directory
Cloning into 'job-postings-fraud'...
remote: Enumerating objects: 232, done.
remote: Counting objects: 100% (232/232), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 232 (delta 84), reused 185 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (232/232), 11.16 MiB | 11.58 MiB/s, done.
Resolving deltas: 100% (84/84), done.
Cloned repo, models at: job-postings-fraud/models/


In [18]:
# ===== LOAD MODELS FROM GITHUB REPO =====
print("\n" + "="*80)
print("LOADING MODELS FROM GITHUB...")
print("="*80)

models_path = "job-postings-fraud/models/"

try:
    # Load individual NB components (pipeline is corrupted)
    print("⚠️ Note: Using individual NB components (nb_pipeline.pkl has issues)")

    with open(f'{models_path}naive_bayes_model.pkl', 'rb') as f:
        nb_model = pickle.load(f)
        print("   ✅ Loaded naive_bayes_model.pkl")

    with open(f'{models_path}vectorizer.pkl', 'rb') as f:
        vectorizer = pickle.load(f)
        print("   ✅ Loaded vectorizer.pkl")

    # Load LSTM tokenizer
    with open(f'{models_path}tokenizer.pkl', 'rb') as f:
        lstm_tokenizer = pickle.load(f)
        print("   ✅ Loaded tokenizer.pkl")

    # Load LSTM model
    lstm_model = load_model(f'{models_path}lstm_model.h5')
    print("   ✅ Loaded lstm_model.h5")

    # Load MiniLM (in subfolder)
    minilm_tokenizer = AutoTokenizer.from_pretrained(f'{models_path}model_miniLM_final/')
    minilm_model = AutoModelForSequenceClassification.from_pretrained(f'{models_path}model_miniLM_final/')
    minilm_model.eval()
    print("   ✅ Loaded model_miniLM_final/")

    print("\n✅ All models loaded successfully!")

except FileNotFoundError as e:
    print(f"❌ Error: File not found - {e}")
    print(f"\nChecking what's in {models_path}:")
    !ls -lh {models_path}

except Exception as e:
    print(f"❌ Error loading models: {e}")
    import traceback
    traceback.print_exc()



LOADING MODELS FROM GITHUB...
⚠️ Note: Using individual NB components (nb_pipeline.pkl has issues)
   ✅ Loaded naive_bayes_model.pkl
   ✅ Loaded vectorizer.pkl


   ✅ Loaded tokenizer.pkl
   ✅ Loaded lstm_model.h5
   ✅ Loaded model_miniLM_final/

✅ All models loaded successfully!


In [19]:
from scipy.sparse import hstack, csr_matrix

def extract_nb_features(df_subset, vectorizer):
    text_features = vectorizer.transform(df_subset['text_processed'])
    numeric = df_subset[['telecommuting',
                         'has_company_logo',
                         'has_questions',
                         'location_fraud_ratio',
                         'character_count']].fillna(0).values
    numeric_sparse = csr_matrix(numeric)
    return hstack([text_features, numeric_sparse])


In [21]:
print("\n" + "="*80)
print("GENERATING PREDICTIONS ON TEST SET...")
print("="*80)

# 1. NB
X_test_nb = extract_nb_features(X_test, vectorizer)
nb_probs = nb_model.predict_proba(X_test_nb)[:, 1]

# 2. LSTM
X_test_lstm = pad_sequences(
    lstm_tokenizer.texts_to_sequences(X_test['text_processed'].tolist()),
    maxlen=200
)
lstm_probs = lstm_model.predict(X_test_lstm, verbose=0).flatten()

# 3. MiniLM
minilm_probs = []
batch_size = 32

for i in range(0, len(X_test), batch_size):
    # Access 'combined_text' from the original df using X_test's index
    batch = df.loc[X_test.iloc[i:i+batch_size].index, 'combined_text'].tolist()
    inputs = minilm_tokenizer(
        batch,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding="max_length",
    )
    with torch.no_grad():
        outputs = minilm_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
        minilm_probs.extend(probs)

minilm_probs = np.array(minilm_probs)

print("NB mean prob:", nb_probs.mean())
print("LSTM mean prob:", lstm_probs.mean())
print("MiniLM mean prob:", minilm_probs.mean())


GENERATING PREDICTIONS ON TEST SET...
NB mean prob: 0.03563280584776362
LSTM mean prob: 0.056169305
MiniLM mean prob: 0.038299665


In [22]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
import numpy as np
import pandas as pd

def evaluate_model(y_true, probs, threshold=0.5, name="Model"):
    preds = (probs >= threshold).astype(int)
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, preds),
        "f1":        f1_score(y_true, preds),
        "precision": precision_score(y_true, preds),
        "recall":    recall_score(y_true, preds),
        "roc_auc":   roc_auc_score(y_true, probs),
    }

print("Individual models:")
for name, probs in [
    ("Naive Bayes", nb_probs),
    ("LSTM",        lstm_probs),
    ("MiniLM+IG",   minilm_probs),
]:
    m = evaluate_model(y_test, probs, name=name)
    print(f"{name:11} | Acc {m['accuracy']:.4f}  F1 {m['f1']:.4f}  "
          f"Prec {m['precision']:.4f}  Rec {m['recall']:.4f}  AUC {m['roc_auc']:.4f}")

# ---------- Ensemble search ----------

def eval_ensemble(y_true, nb_p, lstm_p, mini_p, w_nb, w_lstm, w_minilm, thr=0.5):
    total = w_nb + w_lstm + w_minilm
    if total == 0:
        return None
    w_nb, w_lstm, w_minilm = w_nb/total, w_lstm/total, w_minilm/total
    ens_probs = w_nb*nb_p + w_lstm*lstm_p + w_minilm*mini_p
    preds = (ens_probs >= thr).astype(int)
    return {
        "w_nb": round(w_nb, 3),
        "w_lstm": round(w_lstm, 3),
        "w_minilm": round(w_minilm, 3),
        "accuracy": accuracy_score(y_true, preds),
        "f1":        f1_score(y_true, preds),
        "precision": precision_score(y_true, preds),
        "recall":    recall_score(y_true, preds),
        "roc_auc":   roc_auc_score(y_true, ens_probs),
    }

results = []

# Some hand-picked combos
base_combos = [
    (1, 0, 0),   # NB only
    (0, 1, 0),   # LSTM only
    (0, 0, 1),   # MiniLM only
    (1, 1, 1),   # equal
    (0.5, 0.5, 0),
    (0.2, 0.3, 0.5),
    (0.2, 0.25, 0.55),
    (0.15, 0.15, 0.7),
    (0.1, 0.1, 0.8),
    (0.05, 0.05, 0.9),
]

# Grid around MiniLM-heavy weights
for w_minilm in np.arange(0.5, 1.01, 0.05):
    for w_nb in np.arange(0, 1.01 - w_minilm, 0.05):
        w_lstm = 1 - w_minilm - w_nb
        if w_lstm < 0:
            continue
        base_combos.append((w_nb, w_lstm, w_minilm))

for w_nb, w_lstm, w_minilm in base_combos:
    r = eval_ensemble(y_test, nb_probs, lstm_probs, minilm_probs,
                      w_nb, w_lstm, w_minilm)
    if r:
        results.append(r)

results_df = pd.DataFrame(results).drop_duplicates(
    subset=["w_nb", "w_lstm", "w_minilm"]
)
results_df = results_df.sort_values("f1", ascending=False)

print("\nTop 10 ensembles by F1:")
print(results_df.head(10).to_string(index=False))

best = results_df.iloc[0]
print("\nBest ensemble weights:")
print(f"NB={best['w_nb']:.3f}, LSTM={best['w_lstm']:.3f}, MiniLM={best['w_minilm']:.3f}")
print(f"Accuracy={best['accuracy']:.4f}, F1={best['f1']:.4f}, "
      f"Precision={best['precision']:.4f}, Recall={best['recall']:.4f}, "
      f"AUC={best['roc_auc']:.4f}")


Individual models:
Naive Bayes | Acc 0.9702  F1 0.5960  Prec 0.8676  Rec 0.4538  AUC 0.8494
LSTM        | Acc 0.9737  F1 0.6994  Prec 0.7847  Rec 0.6308  AUC 0.9343
MiniLM+IG   | Acc 0.9748  F1 0.7007  Prec 0.8272  Rec 0.6077  AUC 0.9406

Top 10 ensembles by F1:
 w_nb  w_lstm  w_minilm  accuracy       f1  precision   recall  roc_auc
0.050   0.450     0.500  0.978188 0.729792   0.913295 0.607692 0.971372
0.200   0.300     0.500  0.978001 0.728111   0.908046 0.607692 0.974059
0.100   0.400     0.500  0.978001 0.728111   0.908046 0.607692 0.972687
0.150   0.350     0.500  0.978001 0.728111   0.908046 0.607692 0.973439
0.450   0.050     0.500  0.978188 0.725995   0.928144 0.596154 0.969860
0.400   0.100     0.500  0.978001 0.725581   0.917647 0.600000 0.972978
0.250   0.250     0.500  0.977815 0.725173   0.907514 0.603846 0.974485
0.350   0.150     0.500  0.977815 0.723898   0.912281 0.600000 0.974112
0.000   0.500     0.500  0.977815 0.722611   0.917160 0.596154 0.967090
0.333   0.333    

In [23]:
# Ensemble with best weights
w_nb, w_lstm, w_minilm = 0.05, 0.45, 0.50
total = w_nb + w_lstm + w_minilm
w_nb, w_lstm, w_minilm = w_nb/total, w_lstm/total, w_minilm/total

ensemble_probs = w_nb*nb_probs + w_lstm*lstm_probs + w_minilm*minilm_probs
ensemble_preds = (ensemble_probs >= 0.5).astype(int)

print("Total test samples:", len(y_test))
print("True fraud count:  ", y_test.sum())
print("Pred fraud count:  ", ensemble_preds.sum())
print("Pred legit count:  ", (ensemble_preds == 0).sum())


Total test samples: 5364
True fraud count:   260
Pred fraud count:   173
Pred legit count:   5191


In [24]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, ensemble_preds)
tn, fp, fn, tp = cm.ravel()
print("Confusion matrix:\n", cm)
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")


Confusion matrix:
 [[5089   15]
 [ 102  158]]
TN=5089, FP=15, FN=102, TP=158


In [25]:
fraud_indices = np.where(y_test == 1)[0][:10]
print("Ensemble probs for first 10 fraud cases:")
print(ensemble_probs[fraud_indices])
print("Predictions:", ensemble_preds[fraud_indices])


Ensemble probs for first 10 fraud cases:
[0.5386324  0.0164779  0.91317999 0.97520024 0.23239105 0.92090073
 0.50779738 0.19231928 0.9680196  0.94385281]
Predictions: [1 0 1 1 0 1 1 0 1 1]


In [27]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

w_nb, w_lstm, w_minilm = 0.05, 0.45, 0.50
total = w_nb + w_lstm + w_minilm
w_nb, w_lstm, w_minilm = w_nb/total, w_lstm/total, w_minilm/total

ensemble_probs = w_nb*nb_probs + w_lstm*lstm_probs + w_minilm*minilm_probs

thresholds = np.linspace(0.3, 0.8, 11)  # e.g., 0.30, 0.35, ..., 0.80
rows = []

for thr in thresholds:
    preds = (ensemble_probs >= thr).astype(int)
    rows.append({
        "threshold": thr,
        "accuracy":  accuracy_score(y_test, preds),
        "f1":        f1_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall":    recall_score(y_test, preds),
        "roc_auc":   roc_auc_score(y_test, ensemble_probs),  # same across thresholds
    })

thr_df = pd.DataFrame(rows).sort_values("recall", ascending=False)
print(thr_df.to_string(index=False))


 threshold  accuracy       f1  precision   recall  roc_auc
      0.30  0.979493 0.787645   0.790698 0.784615 0.971372
      0.35  0.981171 0.798403   0.829876 0.769231 0.971372
      0.40  0.981357 0.794239   0.853982 0.742308 0.971372
      0.45  0.976696 0.726477   0.842640 0.638462 0.971372
      0.50  0.978188 0.729792   0.913295 0.607692 0.971372
      0.55  0.975764 0.676617   0.957746 0.523077 0.971372
      0.60  0.974646 0.651282   0.976923 0.488462 0.971372
      0.65  0.974459 0.642298   1.000000 0.473077 0.971372
      0.70  0.973341 0.620690   1.000000 0.450000 0.971372
      0.75  0.972409 0.602151   1.000000 0.430769 0.971372
      0.80  0.971477 0.583106   1.000000 0.411538 0.971372
